# ATTEST — Performance Benchmark

**ATTEST**: Accumulator-based Tamarin-verified Trust for crEdential Systems in IoT  
Built on **COMPASS** (INDOCRYPT 2025) — a lattice-based accumulator for post-quantum IoT credential revocation.

This notebook reproduces the performance numbers from §5 of the ATTEST paper.
It runs on **Google Colab** (free tier) and takes roughly **5–10 minutes** with set2 (the paper parameter set).

### What is measured
| Section | Operations |
|---------|------------|
| §1 | Component micro-benchmarks (BF ops, ring arithmetic) |
| §2 | `VerifyWithRevocation` — baseline and extended deployment |
| §3 | `WitUpdate` — device-side epoch update |
| §4 | Manager `BatchAdd` and `BatchDel` vs batch size |
| §5 | DLT / Relay gate check and epoch log fetch |
| §6 | Wire sizes — witness, ΔZ, BF_Δ, epoch packages |

## Setup

Run this cell once — it clones ATTEST and COMPASS from GitHub and installs dependencies.


In [ ]:
import sys, os

try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/khsjds/ATTEST.git  --quiet
    !git clone https://github.com/khsjds/COMPASS.git --quiet
    sys.path.insert(0, '/content/ATTEST')
    sys.path.insert(0, '/content/COMPASS')

!pip install numpy --quiet

try:
    import attest, compass
    print(f'attest {attest.__version__}  |  compass OK  — setup complete.')
except ImportError as e:
    print(f'Import failed: {e}')
    print('Running locally? Make sure compass/ is on PYTHONPATH and you are in ATTEST/.')


## Configuration

In [ ]:
# ── Parameter sets to benchmark ──────────────────────────────────────────────
PARAM_SETS_TO_RUN = ['set1', 'set2']   # remove one entry to skip it

# ── Accumulator capacity ─────────────────────────────────────────────────────
K_REAL = 10_000   # realistic deployment capacity (determines BF size + BoundZ)
K_POOL = 20       # members actually accumulated in benchmarks

# ── Repetition counts ─────────────────────────────────────────────────────────
# All timeit() calls run warmup iterations first, then N_* measured iterations.
# Reported mean ± std are over these N_* runs — same count for both param sets.
N_VERIFY = 200    # VerifyWithRevocation  (~40 s set1 / ~200 s set2)
N_FAST   = 2000   # sub-ms ops: BF insert/query, ring add, WitUpdate, gate check
N_ADD    = 10     # BatchAdd runs per size (both sets)
N_DEL    = 100    # BatchDel runs per size

# ── Batch sizes for scaling plots ─────────────────────────────────────────────
ADD_SIZES = [1, 5, 10, 20]   # both sets; set1 σ=11,336 makes large m slow
DEL_SIZES = [1, 5, 10, 20]

# ── Estimated total runtime ───────────────────────────────────────────────────
# set2: ~200 s verify + ~3 min BatchAdd + misc        ≈  7 min
# set1: ~40 s verify  + up to ~20 min BatchAdd + misc ≈ 20 min  (tight σ)
# Both sets: up to ~30 min total on Colab free tier.


## Imports

In [ ]:
import time, statistics
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from attest import (
    PARAM_SETS, VC, H_tag, BloomFilter,
    manager_setup, batch_add, batch_del,
    dlt_init, dlt_publish, dlt_gate_check, dlt_fetch_updates,
    wit_update, verify_with_revocation,
)
from attest.params import BF_FP_RATE
from compass.utils import bytes_per_q as _bpq

print('Imports OK.')

## Helpers

In [ ]:
# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
COLORS = {'set1': '#4C72B0', 'set2': '#DD8452'}
LABELS = {'set1': 'set1  (N=512)', 'set2': 'set2  (N=1024)'}

def make_vcs(n, issuer=b'bench'):
    return [VC(issuer_id=issuer, cred_id=f'cred_{i}'.encode(),
               holder_pk=(f'pk_{i}'.encode() * 4)[:32], issued_epoch=0)
            for i in range(n)]

def timeit(fn, n_runs, warmup=3):
    """Warm up, then time n_runs fresh calls. Returns (mean_ms, std_ms)."""
    for _ in range(warmup):
        fn()
    ts = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t0) * 1000)
    return statistics.mean(ts), (statistics.stdev(ts) if n_runs > 1 else 0.0)

print('Helpers ready.')

## Run Benchmarks

Runs all sections for each parameter set in sequence. Progress is printed after each section.

In [ ]:
results = {}

for PSET in PARAM_SETS_TO_RUN:
    sep = chr(0x2500) * 56
    print(f'\n{sep}')
    print(f'  param_set = {PSET}')
    print(sep)
    r = {}

    # ── Fresh world ───────────────────────────────────────────────────────────
    print('  [init] building accumulator ...', end=' ', flush=True)
    mgr  = manager_setup(K=K_REAL, param_set=PSET)  # K_REAL for correct BF/BoundZ
    dlt  = dlt_init(mgr)
    vcs  = make_vcs(K_POOL)
    mgr, witnesses, pkg = batch_add(mgr, vcs)
    dlt  = dlt_publish(dlt, pkg)
    c    = mgr.compass
    print('done')

    # ── §1 Component micro ────────────────────────────────────────────────────
    tag = H_tag(vcs[0])
    bf  = BloomFilter(K=K_POOL, p=BF_FP_RATE); bf.insert(tag)
    Zi, zi, ci1, ci2 = c.deserialize_witness(witnesses[0].data)
    dZ  = pkg.delta_Z;  Z = mgr.Z

    r['bf_insert'] = timeit(lambda: bf.insert(tag),    N_FAST, warmup=20)
    r['bf_query']  = timeit(lambda: bf.query(tag),     N_FAST, warmup=20)
    r['ring_add']  = timeit(lambda: (Zi + dZ) % c.q,   N_FAST, warmup=20)
    r['pf']        = timeit(lambda: c.PF.F(Z),          N_FAST, warmup=10)
    print(f'  [SS1] BF insert={r["bf_insert"][0]*1000:.2f} us'
          f'  query={r["bf_query"][0]*1000:.2f} us'
          f'  ring_add={r["ring_add"][0]*1000:.2f} us'
          f'  PF={r["pf"][0]:.3f} ms')

    # ── §2 VerifyWithRevocation ───────────────────────────────────────────────
    mgr2, del_pkg = batch_del(mgr, [vcs[-1]])
    dlt2 = dlt_publish(dlt, del_pkg)
    bf_local = BloomFilter(K=K_REAL, p=BF_FP_RATE)  # must match del_pkg.bf_delta size
    w0 = witnesses[0]
    for p in dlt_fetch_updates(dlt2, from_epoch=1, to_epoch=dlt2.epoch):
        w0, bf_local = wit_update(c, p, w0, bf_local, vcs[0])

    r['verify_baseline'] = timeit(
        lambda: verify_with_revocation(c, dlt2.Acc, vcs[0], w0, None),
        N_VERIFY, warmup=3)
    r['verify_extended'] = timeit(
        lambda: verify_with_revocation(c, dlt2.Acc, vcs[0], w0, bf_local),
        N_VERIFY, warmup=3)
    print(f'  [SS2] verify baseline={r["verify_baseline"][0]:.2f} ms'
          f'  extended={r["verify_extended"][0]:.2f} ms')

    # ── §3 WitUpdate ─────────────────────────────────────────────────────────
    extra_vcs      = make_vcs(1, issuer=b'extra')
    _, _, add_pkg2 = batch_add(mgr2, extra_vcs)
    ser = bytes(BloomFilter(K=K_REAL, p=BF_FP_RATE).serialize())  # must match K_REAL for merge

    r['wit_base']    = timeit(
        lambda: wit_update(c, add_pkg2, witnesses[0], None, vcs[0]),
        N_FAST, warmup=20)
    r['wit_ext_add'] = timeit(
        lambda: wit_update(c, add_pkg2, witnesses[0],
                           BloomFilter(K=K_REAL, p=BF_FP_RATE, _bits=bytearray(ser)),
                           vcs[0]),
        N_FAST, warmup=20)
    r['wit_ext_rev'] = timeit(
        lambda: wit_update(c, del_pkg, witnesses[0],
                           BloomFilter(K=K_REAL, p=BF_FP_RATE, _bits=bytearray(ser)),
                           vcs[0]),
        N_FAST, warmup=20)
    print(f'  [SS3] WitUpdate base={r["wit_base"][0]*1000:.2f} us'
          f'  ext_rev={r["wit_ext_rev"][0]*1000:.2f} us')

    # ── §4 BatchAdd scaling ───────────────────────────────────────────────────
    ms_to_run = ADD_SIZES
    n_add     = N_ADD   # same repetition count for both sets
    print(f'  [SS4] BatchAdd (sizes={ms_to_run}, n_runs={n_add}) ...')
    r['batch_add'] = {}
    for m in ms_to_run:
        new_vcs = make_vcs(m, issuer=f'a{m}{PSET}'.encode())
        mt, st  = timeit(lambda nv=new_vcs: batch_add(mgr, nv), n_add, warmup=0)
        r['batch_add'][m] = (mt, st)
        print(f'         m={m}: {mt:.1f} ms total  ({mt/m:.1f} ms/member)')

    # ── §4 BatchDel scaling ───────────────────────────────────────────────────
    print(f'  [SS4] BatchDel (sizes={DEL_SIZES}, n_runs={N_DEL}) ...')
    r['batch_del'] = {}
    for l in DEL_SIZES:
        dv = vcs[:l]
        mt, st = timeit(lambda dv=dv: batch_del(mgr, dv), N_DEL, warmup=5)
        r['batch_del'][l] = (mt, st)
        print(f'         l={l}: {mt*1000:.1f} us')

    # ── §5 DLT ops ───────────────────────────────────────────────────────────
    r['gate_allow'] = timeit(lambda: dlt_gate_check(dlt2, vcs[0]),  N_FAST, warmup=20)
    r['gate_deny']  = timeit(lambda: dlt_gate_check(dlt2, vcs[-1]), N_FAST, warmup=20)
    r['dlt_fetch']  = timeit(
        lambda: dlt_fetch_updates(dlt2, from_epoch=1, to_epoch=2), N_FAST, warmup=20)
    print(f'  [SS5] gate allow={r["gate_allow"][0]*1000:.2f} us'
          f'  deny={r["gate_deny"][0]*1000:.2f} us')

    # ── §6 Wire sizes ─────────────────────────────────────────────────────────
    bpq = _bpq(c.q)
    r['wire'] = {
        'Witness'       : len(witnesses[0].data),
        'Delta-Z'       : len(pkg.delta_Z) * bpq,
        'BF-delta'      : BloomFilter(K=10_000, p=BF_FP_RATE).size_bytes(),
        'Add broadcast' : len(pkg.delta_Z) * bpq + len(pkg.sig),
        'Del broadcast' : (len(pkg.delta_Z) * bpq
                           + BloomFilter(K=10_000, p=BF_FP_RATE).size_bytes()
                           + len(pkg.sig)),
    }

    results[PSET] = r
    print(f'  {PSET} complete.')

print('\nAll benchmarks done.')

## Plots

All figures are also saved as PNG in the current directory.

In [ ]:
# ── Fig 1: VerifyWithRevocation ───────────────────────────────────────────────
psets  = [p for p in ['set1', 'set2'] if p in results]
labels = ['Baseline\n(accum. only)', 'Extended\n(BF + accum.)']
keys   = ['verify_baseline', 'verify_extended']
x      = np.arange(len(labels))
width  = 0.35

fig, ax = plt.subplots(figsize=(6, 4))
for i, ps in enumerate(psets):
    r     = results[ps]
    means = [r[k][0] for k in keys]
    errs  = [r[k][1] for k in keys]
    ax.bar(x + (i - len(psets)/2 + 0.5) * width, means, width,
           label=LABELS[ps], color=COLORS[ps],
           yerr=errs, capsize=4, error_kw={'linewidth':1.2})

ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Time (ms)')
ax.set_title('VerifyWithRevocation')
ax.legend()
plt.tight_layout()
plt.savefig('fig_verify.png', bbox_inches='tight')
plt.show()
print('Saved: fig_verify.png')

In [ ]:
# ── Fig 2: WitUpdate ─────────────────────────────────────────────────────────
psets  = [p for p in ['set1', 'set2'] if p in results]
labels = ['Baseline', 'Extended\n(add epoch)', 'Extended\n(revoke epoch)']
keys   = ['wit_base', 'wit_ext_add', 'wit_ext_rev']
x      = np.arange(len(labels))
width  = 0.35

fig, ax = plt.subplots(figsize=(6, 4))
for i, ps in enumerate(psets):
    r     = results[ps]
    means = [r[k][0] * 1000 for k in keys]   # ms -> us
    errs  = [r[k][1] * 1000 for k in keys]
    ax.bar(x + (i - len(psets)/2 + 0.5) * width, means, width,
           label=LABELS[ps], color=COLORS[ps],
           yerr=errs, capsize=4, error_kw={'linewidth':1.2})

ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Time (us)')
ax.set_title('WitUpdate (device-side epoch update)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_witupdate.png', bbox_inches='tight')
plt.show()
print('Saved: fig_witupdate.png')

In [ ]:
# ── Fig 3: BatchAdd scaling ───────────────────────────────────────────────────
psets = [p for p in ['set1', 'set2'] if p in results]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

for ps in psets:
    r   = results[ps]
    ms  = sorted(r['batch_add'].keys())
    tot = [r['batch_add'][m][0]   for m in ms]
    ter = [r['batch_add'][m][1]   for m in ms]
    ppm = [r['batch_add'][m][0]/m for m in ms]
    per = [r['batch_add'][m][1]/m for m in ms]
    kw  = dict(marker='o', label=LABELS[ps], color=COLORS[ps], capsize=4)
    ax1.errorbar(ms, tot, yerr=ter, **kw)
    ax2.errorbar(ms, ppm, yerr=per, **kw)

for ax, title, ylabel in [
    (ax1, 'BatchAdd — total time',    'Total time (ms)'),
    (ax2, 'BatchAdd — per member',    'Time per member (ms)'),
]:
    ax.set_xlabel('Batch size m')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.savefig('fig_batch_add.png', bbox_inches='tight')
plt.show()
print('Saved: fig_batch_add.png')
print('Note: set1 n_add=1 (single run, high variance) — parallelisable across members in a C implementation')

In [ ]:
# ── Fig 4: BatchDel scaling ───────────────────────────────────────────────────
psets = [p for p in ['set1', 'set2'] if p in results]
fig, ax = plt.subplots(figsize=(6, 4))

for ps in psets:
    r   = results[ps]
    ls  = sorted(r['batch_del'].keys())
    mus = [r['batch_del'][l][0] * 1000 for l in ls]   # ms -> us
    err = [r['batch_del'][l][1] * 1000 for l in ls]
    ax.errorbar(ls, mus, yerr=err, marker='s',
                label=LABELS[ps], color=COLORS[ps], capsize=4)

ax.set_xlabel('Revocation count l')
ax.set_ylabel('Time (us)')
ax.set_title('BatchDel — deterministic ring subtraction')
ax.legend()
ax.set_xticks(DEL_SIZES)
plt.tight_layout()
plt.savefig('fig_batch_del.png', bbox_inches='tight')
plt.show()
print('Saved: fig_batch_del.png')

In [ ]:
# ── Fig 5: Wire / storage sizes ───────────────────────────────────────────────
psets     = [p for p in ['set1', 'set2'] if p in results]
wire_keys = list(results[psets[0]]['wire'].keys())

fig, ax = plt.subplots(figsize=(8, 4))
y     = np.arange(len(wire_keys))
width = 0.35

for i, ps in enumerate(psets):
    vals = [results[ps]['wire'][k] / 1024 for k in wire_keys]
    bars = ax.barh(y + (i - len(psets)/2 + 0.5) * width, vals, width,
                   label=LABELS[ps], color=COLORS[ps])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f'{v:.1f} KB', va='center', fontsize=9)

ax.set_yticks(y); ax.set_yticklabels(wire_keys)
ax.set_xlabel('Size (KB)')
ax.set_title('Wire and storage sizes')
ax.legend()
plt.tight_layout()
plt.savefig('fig_wire_sizes.png', bbox_inches='tight')
plt.show()
print('Saved: fig_wire_sizes.png')

## Summary Table

In [ ]:
import platform
print(f'ATTEST Performance Summary')
print(f'Python {platform.python_version()}  /  {platform.processor() or platform.machine()}')

for ps_name in PARAM_SETS_TO_RUN:
    if ps_name not in results: continue
    r  = results[ps_name]
    ps = PARAM_SETS[ps_name]
    print(f'\n  ── {ps_name.upper()}  N={ps.N}  q={ps.q} ──')
    print(f'  {"Operation":<40}  {"Mean":>10}  {"Std":>8}')
    print('  ' + '-' * 62)
    for lbl, key, scale, unit in [
        ('VerifyWithRevocation (baseline)',   'verify_baseline',   1,    'ms'),
        ('VerifyWithRevocation (extended)',   'verify_extended',   1,    'ms'),
        ('WitUpdate (baseline)',              'wit_base',          1000, 'us'),
        ('WitUpdate (extended, revoke)',      'wit_ext_rev',       1000, 'us'),
        ('DLT gate check (allow)',            'gate_allow',        1000, 'us'),
        ('DLT gate check (deny)',             'gate_deny',         1000, 'us'),
    ]:
        m, s = r[key]
        print(f'  {lbl:<40}  {m*scale:>8.2f} {unit}  {s*scale:>6.2f}')
    m_b, _ = r['verify_baseline']
    print(f'  {"Throughput (baseline)":<40}  {1000/m_b:>8.1f} ver/s')
    for m_size in sorted(r['batch_add'].keys()):
        mt, st = r['batch_add'][m_size]
        print(f'  BatchAdd m={m_size:<34}  {mt:>8.1f} ms  {st:>6.1f}  ({mt/m_size:.1f} ms/member)')
    for l_size in DEL_SIZES:
        mt, st = r['batch_del'][l_size]
        print(f'  BatchDel l={l_size:<34}  {mt*1000:>8.1f} us  {mt*1000:>6.1f}')
    print()
    print(f'  {"Component":<40}  {"Bytes":>7}    KB')
    print('  ' + '-' * 62)
    for k, v in r['wire'].items():
        print(f'  {k:<40}  {v:>7}  {v/1024:>5.1f} KB')

## LaTeX Data Export

Writes `attest_bench_data.tex` with pgfplots-ready `\def` macros — the same approach used in the COMPASS paper. Include this file in the paper with `\input{attest_bench_data}` and reference the macros directly in `\addplot coordinates` blocks.

In [ ]:
# ── Export pgfplots-ready LaTeX data (COMPASS style) ─────────────────────────
# Include in paper with: \input{attest_bench_data}
# Use in figures: \addplot+[error bars/.cd,y dir=both,y explicit]
#                   coordinates {\attsetoneVerify};

lines = [
    '% ATTEST benchmark data — generated by attest_benchmark.ipynb',
    '% Format: (x, mean) +- (0, std)   [x is batch size m or l; for scalars x=1]',
    '%',
]

def pgf_scalar(name, pset, key, scale=1.0, comment=''):
    """Single-value metric as a 1-point coordinate sequence."""
    m, s = results[pset][key]
    coord = f'(1, {m*scale:.4f}) +- (0, {s*scale:.4f})'
    c = f'  % {comment}' if comment else ''
    return f'\\def\\att{pset.capitalize()}{name}{{{coord}}}{c}'

def pgf_series(name, pset, key_dict, scale=1.0, comment=''):
    """Scaling series (e.g. batch_add) as multi-point coordinate sequence."""
    coords = ' '.join(
        f'({x}, {m*scale:.4f}) +- (0, {s*scale:.4f})'
        for x, (m, s) in sorted(key_dict.items())
    )
    c = f'  % {comment}' if comment else ''
    return f'\\def\\att{pset.capitalize()}{name}{{{coords}}}{c}'

for pset in PARAM_SETS_TO_RUN:
    if pset not in results:
        continue
    r = results[pset]
    P = pset.capitalize()  # Set1 / Set2
    lines.append(f'')
    lines.append(f'% ── {P} ─────────────────────────────────────────────────')

    # Scalar metrics (ms)
    lines.append(pgf_scalar('VerifyBaseline', pset, 'verify_baseline',  comment='ms'))
    lines.append(pgf_scalar('VerifyExtended', pset, 'verify_extended',  comment='ms'))
    lines.append(pgf_scalar('WitUpdateBase',  pset, 'wit_base',  scale=1000, comment='us'))
    lines.append(pgf_scalar('WitUpdateRev',   pset, 'wit_ext_rev', scale=1000, comment='us'))
    lines.append(pgf_scalar('GateAllow',      pset, 'gate_allow', scale=1000, comment='us'))
    lines.append(pgf_scalar('GateDeny',       pset, 'gate_deny',  scale=1000, comment='us'))

    # BatchAdd scaling (ms, per whole batch)
    lines.append(pgf_series('BatchAdd', pset, r['batch_add'], comment='ms total; x=batch size m'))

    # BatchAdd per-member (ms)
    per_member = {x: (m/x, s/x) for x, (m, s) in r['batch_add'].items()}
    lines.append(pgf_series('BatchAddPerMember', pset, per_member, comment='ms per member'))

    # BatchDel scaling (ms)
    batch_del_ms = {x: (m*1000, s*1000) for x, (m, s) in r['batch_del'].items()}
    lines.append(pgf_series('BatchDel', pset, batch_del_ms, comment='ms total; x=revocations l'))

    # Wire sizes (bytes) — stored as scalar (x=1) but labelled
    lines.append('%')
    for wk, wv in r['wire'].items():
        safe = wk.replace(' ', '').replace('(', '').replace(')', '').replace('/', 'Or').replace('+', 'Plus')
        lines.append(f'\\def\\att{P}Wire{safe}{{{wv}}}  % bytes')

out = '\n'.join(lines) + '\n'
with open('attest_bench_data.tex', 'w') as f:
    f.write(out)

print('Written: attest_bench_data.tex')
print()
# Preview first 30 lines
for line in out.split('\n')[:30]:
    print(line)